In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Masking, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Subtract, Multiply, concatenate
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Kaggle Environment Setup
import os

DATASET_SLUG = "siamese-data"   # <-- GANTI sesuai nama dataset kamu
DATA_DIR     = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR      = "/kaggle/working"

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

for fname in ['questions_emb.npy', 'answerkeys_emb.npy', 'answers_emb.npy', 'metadata.pkl']:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<25} -> {status}")


In [ ]:
# Load embeddings
questions_emb  = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
answerkeys_emb = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))

# Load metadata (IDJwb, IDPSJ, grade)
metadata = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

print("=== Hasil Load ===")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"answers_emb    : {answers_emb.shape}")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())


=== Hasil Load ===
questions_emb  : (817, 135, 300)
answerkeys_emb : (817, 90, 300)
answers_emb    : (817, 80, 300)

Metadata       : 817 rows
Kolom metadata : ['IDJwb', 'IDPSJ', 'grade']

IDPSJ unik     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Distribusi grade:
grade
1      89
2      53
3      65
4      49
5      53
6      87
7      75
8      84
9      39
10    223
Name: count, dtype: int64


In [ ]:
ENC_DIM      = 256   # bilstm_units=128 → Bidirectional output = 256D
BILSTM_UNITS = 128


def build_joint_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                      bilstm_units=128, dropout=0.3):
    """
    Joint model dengan 3 input: question, answerkey, answer.
    Dilatih end-to-end; regression head berfungsi sebagai sinyal
    training agar encoder menghasilkan vektor yang bermakna.

    Encoder:
      bilstm_question : khusus question
      shared_bilstm   : dipakai answerkey dan answer (Siamese)

    Fitur: [eq, ea, eak, |eak-ea|, eak⊙ea]  →  5 × 256D = 1280D
    """
    bilstm_q      = Bidirectional(
        LSTM(bilstm_units, return_sequences=False), name='bilstm_question'
    )
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=False), name='shared_bilstm'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    msk_q  = Masking(mask_value=0.0, name='msk_q')(inp_q)
    msk_ak = Masking(mask_value=0.0, name='msk_ak')(inp_ak)
    msk_a  = Masking(mask_value=0.0, name='msk_a')(inp_a)

    eq  = bilstm_q(msk_q)
    eak = shared_bilstm(msk_ak)
    ea  = shared_bilstm(msk_a)

    diff     = Subtract(name='diff')([eak, ea])
    abs_diff = Lambda(lambda x: tf.abs(x), name='abs_diff')(diff)
    had_prod = Multiply(name='had_prod')([eak, ea])

    merged = concatenate([eq, ea, eak, abs_diff, had_prod], name='merged')

    x   = Dense(256, activation='relu', name='dense_256')(merged)
    x   = Dropout(dropout,              name='dropout')(x)
    x   = Dense(64,  activation='relu', name='dense_64')(x)
    out = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out, name='joint_model')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


def make_encoder_model(joint_model, seq_len, emb_dim, layer_name='shared_bilstm'):
    """Ekstrak sub-model encoder dari joint_model yang sudah dilatih."""
    inp = Input(shape=(seq_len, emb_dim))
    msk = Masking(mask_value=0.0)(inp)
    enc = joint_model.get_layer(layer_name)(msk)
    return Model(inputs=inp, outputs=enc)


_tmp = build_joint_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2]
)
_tmp.summary()


In [ ]:
# LOPO (Leave-One-Participant-Out)
# Per fold: latih joint model → ekstrak encoder → encode semua split → simpan .npz

idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_all      = metadata['grade'].values.astype(np.float32)

bilstm_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}  |  Train={train_ids}")

    train_idx = metadata[metadata['IDPSJ'].isin(train_ids)].index.values
    val_idx   = metadata[metadata['IDPSJ'] == val_id].index.values
    test_idx  = metadata[metadata['IDPSJ'] == test_id].index.values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Data  ->  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_train  = get_split(questions_emb,  train_idx)
    X_ak_train = get_split(answerkeys_emb, train_idx)
    X_a_train  = get_split(answers_emb,    train_idx)
    X_q_val    = get_split(questions_emb,  val_idx)
    X_ak_val   = get_split(answerkeys_emb, val_idx)
    X_a_val    = get_split(answers_emb,    val_idx)
    X_q_test   = get_split(questions_emb,  test_idx)
    X_ak_test  = get_split(answerkeys_emb, test_idx)
    X_a_test   = get_split(answers_emb,    test_idx)

    # Training
    joint_model = build_joint_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        bilstm_units = BILSTM_UNITS,
        dropout      = 0.3
    )

    es = EarlyStopping(monitor='val_loss', patience=5,
                       restore_best_weights=True, verbose=0)

    joint_model.fit(
        [X_q_train, X_ak_train, X_a_train], y_train,
        validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
        epochs=50, batch_size=32,
        callbacks=[es], verbose=1
    )

    # Ekstrak encoder
    encoder_q  = make_encoder_model(joint_model,
                                    questions_emb.shape[1], questions_emb.shape[2],
                                    layer_name='bilstm_question')
    encoder_ak = make_encoder_model(joint_model,
                                    answerkeys_emb.shape[1], answerkeys_emb.shape[2],
                                    layer_name='shared_bilstm')
    encoder_a  = make_encoder_model(joint_model,
                                    answers_emb.shape[1], answers_emb.shape[2],
                                    layer_name='shared_bilstm')

    # Encode semua split → 256D
    eq_train  = encoder_q.predict(X_q_train,   verbose=0)
    eq_val    = encoder_q.predict(X_q_val,     verbose=0)
    eq_test   = encoder_q.predict(X_q_test,    verbose=0)

    eak_train = encoder_ak.predict(X_ak_train, verbose=0)
    eak_val   = encoder_ak.predict(X_ak_val,   verbose=0)
    eak_test  = encoder_ak.predict(X_ak_test,  verbose=0)

    ea_train  = encoder_a.predict(X_a_train,   verbose=0)
    ea_val    = encoder_a.predict(X_a_val,     verbose=0)
    ea_test   = encoder_a.predict(X_a_test,    verbose=0)

    print(f"  Encoded  ->  eq: {eq_train.shape}  |  eak: {eak_train.shape}  |  ea: {ea_train.shape}")

    # Simpan per-fold
    fold_path = os.path.join(OUT_DIR, f'vectors_fold_{i+1:02d}.npz')
    np.savez(
        fold_path,
        eq_train=eq_train,   eak_train=eak_train,   ea_train=ea_train,   y_train=y_train,
        eq_val=eq_val,       eak_val=eak_val,        ea_val=ea_val,       y_val=y_val,
        eq_test=eq_test,     eak_test=eak_test,      ea_test=ea_test,     y_test=y_test,
        test_idpsj=np.array([test_id]),
        val_idpsj=np.array([val_id])
    )
    print(f"  Saved  ->  {fold_path}")

    bilstm_results.append({
        'fold': i + 1, 'test_idpsj': test_id, 'val_idpsj': val_id,
        'n_train': len(y_train), 'n_val': len(y_val), 'n_test': len(y_test)
    })

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")


In [ ]:
results_df = pd.DataFrame(bilstm_results)

print("=== Ringkasan Encoding per Fold ===")
print(results_df[['fold','test_idpsj','val_idpsj',
                   'n_train','n_val','n_test']].to_string(index=False))

print(f"\n=== File Tersimpan di {OUT_DIR} ===")
for fold_num in results_df['fold']:
    fpath = os.path.join(OUT_DIR, f'vectors_fold_{fold_num:02d}.npz')
    size  = os.path.getsize(fpath) / 1024
    print(f"  vectors_fold_{fold_num:02d}.npz  ({size:.1f} KB)")
